# 01 — Master Data

Generates and writes all dimension tables.
Run this notebook before any others.

In [ ]:
%run ./00_helpers

In [ ]:
# Parameters — override these values from a Fabric pipeline or pass via mssparkutils.notebook.run()
RANDOM_SEED       = 42
SCHEMA_NAME       = "rockline"
NUM_PRODUCTS      = 600
NUM_SUPPLIERS     = 50
NUM_CUSTOMERS     = 3500
TRADE_CUSTOMER_PCT = 0.75
NUM_EMPLOYEES     = 340

In [ ]:
from faker import Faker
import random
from datetime import datetime, date, timedelta
from decimal import Decimal

rng  = random.Random(RANDOM_SEED)
fake = Faker("en_GB")
Faker.seed(RANDOM_SEED)

_NOW = datetime.utcnow()
_LOAD_DATE = _NOW.date()

## Branches

In [ ]:
branch_rows = []
for b in BRANCHES:
    branch_rows.append((
        b["branch_id"], b["branch_code"], b["branch_name"],
        b["is_hq"], b["is_distribution_hub"],
        b["address_line_1"], b["address_line_2"], b["town"], b["postcode"],
        b["phone"], b["email"], b["manager_name"],
        b["opening_mon_fri"], b["opening_saturday"], b["opening_sunday"],
        _NOW,
    ))

dim_branch_df = to_spark_df(branch_rows, SCHEMA_DIM_BRANCH)
dim_branch_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_branch")
print(f"dim_branch: {dim_branch_df.count()} rows")

## Product Categories

In [ ]:
cat_rows = [(c["category_id"], c["category_code"], c["category_name"],
             c["sku_prefix"], c["approx_sku_count"]) for c in CATEGORIES]

dim_cat_df = to_spark_df(cat_rows, SCHEMA_DIM_PRODUCT_CATEGORY)
dim_cat_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_product_category")
print(f"dim_product_category: {dim_cat_df.count()} rows")

## Suppliers

In [ ]:
_CONSTRUCTION_SUFFIXES = [" Building Supplies Ltd", " Materials Group plc",
    " Industrial Supplies", " Wholesale Ltd", " Distribution Ltd",
    " Steel Stockholders", " Timber Merchants", " Builders Merchants"]

supplier_rows = []
for i in range(1, NUM_SUPPLIERS + 1):
    cat = CATEGORIES[(i - 1) % len(CATEGORIES)]
    company_raw = fake.company()
    company = company_raw + rng.choice(_CONSTRUCTION_SUFFIXES)
    supplier_rows.append((
        i,
        company,
        fake.name(),
        random_uk_phone(rng),
        f"sales@{re.sub(r'[^a-z0-9]', '', company_raw.lower()[:12])}.co.uk",
        fake.street_address(),
        fake.city(),
        fake.postcode(),
        rng.choice([30, 30, 60]),
        rng.randint(2, 21),
        cat["category_id"],
        True,
        _NOW,
    ))

dim_supplier_df = to_spark_df(supplier_rows, SCHEMA_DIM_SUPPLIER)
dim_supplier_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_supplier")
print(f"dim_supplier: {dim_supplier_df.count()} rows")

## Products

In [ ]:
# Distribute remaining slots proportionally across categories
total_approx = sum(c["approx_sku_count"] for c in CATEGORIES)
seed_count   = len(SEED_PRODUCTS)
gen_count    = max(0, NUM_PRODUCTS - seed_count)

cat_alloc = {}
remaining = gen_count
for i, cat in enumerate(CATEGORIES):
    if i == len(CATEGORIES) - 1:
        cat_alloc[cat["category_id"]] = remaining
    else:
        alloc = round(gen_count * cat["approx_sku_count"] / total_approx)
        cat_alloc[cat["category_id"]] = alloc
        remaining -= alloc

# Build supplier lookup: category_id -> list of supplier_ids
sup_by_cat = {}
for row in supplier_rows:
    sup_by_cat.setdefault(row[10], []).append(row[0])

# Track highest SKU number per prefix to avoid collisions with seed SKUs
sku_counters = {}
for p in SEED_PRODUCTS:
    prefix = p["sku"].split("-")[0]
    num = int(p["sku"].split("-")[1])
    sku_counters[prefix] = max(sku_counters.get(prefix, 0), num)

product_rows = []
pid = 1

# Emit seed products first
for p in SEED_PRODUCTS:
    sups = sup_by_cat.get(p["category_id"], [1])
    cost  = Decimal(str(p["standard_cost_gbp"]))
    lp    = Decimal(str(round(float(cost) * rng.uniform(1.25, 1.60), 2)))
    tp    = Decimal(str(round(float(lp)   * rng.uniform(0.72, 0.88), 2)))
    product_rows.append((
        pid, p["sku"], p["product_name"], p["category_id"],
        p["unit_of_measure"], p.get("pack_size_description"),
        cost, lp, tp,
        Decimal(str(round(rng.uniform(0.5, 80.0), 2))),
        True, p["is_stocked_at_hub_only"],
        rng.choice(sups),
        _NOW,
    ))
    pid += 1

# Generate remaining products
for cat in CATEGORIES:
    cid = cat["category_id"]
    prefix = cat["sku_prefix"]
    sups = sup_by_cat.get(cid, [1])
    counter = sku_counters.get(prefix, 0)
    for _ in range(cat_alloc.get(cid, 0)):
        counter += 1
        # skip any SKU number already used by a seed product
        sku = make_sku(prefix, counter)
        while sku in _SEED_SKUS:
            counter += 1
            sku = make_sku(prefix, counter)
        name  = generate_product_name(cid, rng)
        uom   = generate_product_uom(cid, rng)
        cost  = Decimal(str(generate_product_cost(cid, rng)))
        lp    = Decimal(str(round(float(cost) * rng.uniform(1.25, 1.60), 2)))
        tp    = Decimal(str(round(float(lp)   * rng.uniform(0.72, 0.88), 2)))
        hub_only = rng.random() < 0.08
        product_rows.append((
            pid, sku, name, cid, uom, None,
            cost, lp, tp,
            Decimal(str(round(rng.uniform(0.1, 120.0), 2))),
            True, hub_only,
            rng.choice(sups),
            _NOW,
        ))
        pid += 1

dim_product_df = to_spark_df(product_rows, SCHEMA_DIM_PRODUCT)
dim_product_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_product")
print(f"dim_product: {dim_product_df.count()} rows")

## Customers

In [ ]:
_CONSTRUCTION_COMPANY_SUFFIXES = [
    " Construction Ltd", " Building Contractors", " Civil Engineering Ltd",
    " Groundworks Ltd", " Roofing & Cladding Ltd", " Mechanical & Electrical",
    " Property Developments Ltd", " Building Services", " Contractors Ltd",
    " Builders", " Developments plc",
]
_CREDIT_LIMITS = [5000, 10000, 25000, 50000, 100000]
_CREDIT_WEIGHTS = [0.30, 0.35, 0.20, 0.10, 0.05]

n_trade   = round(NUM_CUSTOMERS * TRADE_CUSTOMER_PCT)
n_private = NUM_CUSTOMERS - n_trade

# branch weights: HQ 20%, others 16% each
branch_ids  = [b["branch_id"] for b in BRANCHES]
branch_wts  = [0.20, 0.16, 0.16, 0.16, 0.16, 0.16]

customer_rows = []

for i in range(1, n_trade + 1):
    branch_id = rng.choices(branch_ids, weights=branch_wts)[0]
    first, last = fake.first_name(), fake.last_name()
    company = fake.company() + rng.choice(_CONSTRUCTION_COMPANY_SUFFIXES)
    credit  = Decimal(str(rng.choices(_CREDIT_LIMITS, weights=_CREDIT_WEIGHTS)[0]))
    created = _LOAD_DATE - timedelta(days=rng.randint(30, 4000))
    customer_rows.append((
        i, "trade", company, first, last,
        trade_email(company), random_uk_phone(rng),
        fake.street_address(), fake.city(), fake.postcode(),
        f"TRD-{i:05d}",
        credit, 30, branch_id,
        None,   # account_manager_id assigned after employees exist
        True, datetime.combine(created, datetime.min.time()),
    ))

for j in range(1, n_private + 1):
    cid = n_trade + j
    branch_id = rng.choice(branch_ids)
    first, last = fake.first_name(), fake.last_name()
    created = _LOAD_DATE - timedelta(days=rng.randint(10, 2000))
    customer_rows.append((
        cid, "private", None, first, last,
        fake.email(), random_uk_phone(rng),
        fake.street_address(), fake.city(), fake.postcode(),
        f"PVT-{j:05d}",
        None, 0, branch_id, None,
        True, datetime.combine(created, datetime.min.time()),
    ))

dim_customer_df = to_spark_df(customer_rows, SCHEMA_DIM_CUSTOMER)
dim_customer_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_customer")
print(f"dim_customer: {dim_customer_df.count()} rows")

## Employees

In [ ]:
from decimal import Decimal

_SALARY_BANDS = {
    "Chief Executive Officer":   (120000, 200000),
    "Chief Operating Officer":   (100000, 160000),
    "Finance Director":          (90000,  140000),
    "Head of Procurement":       (65000,  95000),
    "Head of Sales":             (65000,  95000),
    "Head of Logistics":         (60000,  90000),
    "IT & Digital Manager":      (50000,  75000),
    "HR Manager":                (45000,  65000),
    "Branch Manager":            (42000,  58000),
    "Assistant Branch Manager":  (32000,  42000),
    "Account Manager":           (28000,  38000),
    "Trade Counter Sales Advisor":(22000, 30000),
    "Purchasing Officer":        (28000,  38000),
    "Finance Clerk":             (22000,  30000),
    "IT Technician":             (28000,  38000),
    "HR Coordinator":            (25000,  35000),
    "Yard Supervisor":           (24000,  32000),
    "Warehouse Operative":       (20000,  27000),
    "Forklift Driver":           (22000,  29000),
    "HGV Driver":                (28000,  38000),
}

def salary(title, rng):
    lo, hi = _SALARY_BANDS.get(title, (20000, 35000))
    return Decimal(str(round(rng.uniform(lo, hi), -2)))

def hire_date(seniority, rng):
    if seniority == "senior":
        years_ago = rng.randint(10, 37)
    elif seniority == "mid":
        years_ago = rng.randint(3, 15)
    else:
        years_ago = rng.randint(0, 5)
    d = _LOAD_DATE - timedelta(days=years_ago * 365 + rng.randint(0, 364))
    return max(d, date(1988, 1, 1))

employee_rows = []
eid = 1
_NOW_TS = _NOW

# ── HQ Leadership (all at branch_id=1) ──────────────────────────────
for (first, last, title, dept) in LEADERSHIP:
    employee_rows.append((
        eid, f"EMP-{eid:05d}", first, last, title, dept,
        1, hire_date("senior", rng),
        salary(title, rng), True,
        employee_email(first, last), None, _NOW_TS,
    ))
    eid += 1

# ── Branch Managers (1 per branch, real names from website) ─────────
branch_manager_ids = {}
for b in BRANCHES:
    parts = b["manager_name"].split(" ", 1)
    first, last = parts[0], parts[1] if len(parts) > 1 else "Manager"
    branch_manager_ids[b["branch_id"]] = eid
    employee_rows.append((
        eid, f"EMP-{eid:05d}", first, last, "Branch Manager", "Management",
        b["branch_id"], hire_date("senior", rng),
        salary("Branch Manager", rng), True,
        employee_email(first, last), 2,  # reports to COO
        _NOW_TS,
    ))
    eid += 1

# ── Branch headcount targets ─────────────────────────────────────────
_BRANCH_TARGETS = {1: 90, 2: 55, 3: 45, 4: 50, 5: 50, 6: 40}
_BRANCH_ROLES = [
    ("Assistant Branch Manager",   "Management",  1, "mid"),
    ("Account Manager",            "Sales",        3, "mid"),
    ("Trade Counter Sales Advisor","Sales",        6, "entry"),
    ("Yard Supervisor",            "Warehouse",    1, "mid"),
    ("Warehouse Operative",        "Warehouse",    8, "entry"),
    ("Forklift Driver",            "Warehouse",    2, "entry"),
    ("HGV Driver",                 "Logistics",    3, "mid"),
]
_HQ_EXTRA_ROLES = [
    ("Purchasing Officer","Purchasing", 3, "mid"),
    ("Finance Clerk",     "Finance",    4, "entry"),
    ("IT Technician",     "IT",         2, "mid"),
    ("HR Coordinator",    "HR",         2, "entry"),
]

already_added = 1 + len(LEADERSHIP)  # leadership + branch managers

for b in BRANCHES:
    bid = b["branch_id"]
    target = _BRANCH_TARGETS.get(bid, 45) - 1  # -1 for branch manager already added
    added = 0
    roles = _BRANCH_ROLES + (_HQ_EXTRA_ROLES if bid == 1 else [])
    mgr_id = branch_manager_ids[bid]

    for (title, dept, count, seniority) in roles:
        for _ in range(count):
            if added >= target:
                break
            first, last = fake.first_name(), fake.last_name()
            employee_rows.append((
                eid, f"EMP-{eid:05d}", first, last, title, dept,
                bid, hire_date(seniority, rng),
                salary(title, rng), True,
                employee_email(first, last), mgr_id,
                _NOW_TS,
            ))
            eid += 1
            added += 1
        if added >= target:
            break

dim_employee_df = to_spark_df(employee_rows, SCHEMA_DIM_EMPLOYEE)
dim_employee_df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_employee")
print(f"dim_employee: {dim_employee_df.count()} rows")

# ── Back-fill account_manager_id on trade customers ──────────────────
sales_roles  = {"Account Manager", "Trade Counter Sales Advisor"}
sales_emps   = [(r[0], r[6]) for r in employee_rows if r[3] in sales_roles or r[4] in sales_roles]

def pick_am(branch_id):
    candidates = [e[0] for e in sales_emps if e[1] == branch_id]
    return rng.choice(candidates) if candidates else None

updated_customers = []
for row in customer_rows:
    if row[1] == "trade":
        am = pick_am(row[13])
        updated_customers.append(row[:14] + (am,) + row[15:])
    else:
        updated_customers.append(row)

dim_customer_df2 = to_spark_df(updated_customers, SCHEMA_DIM_CUSTOMER)
dim_customer_df2.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA_NAME}.dim_customer")
print(f"dim_customer (with account managers): {dim_customer_df2.count()} rows")

## Summary

In [ ]:
for tbl in ["dim_branch", "dim_product_category", "dim_supplier",
            "dim_product", "dim_customer", "dim_employee"]:
    cnt = spark.table(f"{SCHEMA_NAME}.{tbl}").count()
    print(f"  {tbl:<30} {cnt:>6} rows")